# 🩺 DR — TRUE 1st Place Kaggle Replication v23
## 8-Model Ensemble | Regression + SmoothL1 | GeM + EMA | Batch-Level Resume
### Target QWK ≈ 0.93–0.936 | RTX 2050 CUDA + Apple MPS + CPU
---


## Step 0 — Global Resume System


In [1]:
import os,json,pickle,time,sys,platform,subprocess,shutil
from pathlib import Path
HOME=Path.home(); BASE=Path(os.environ.get("DR_BASE",str(HOME/"DR_data")))
DATA=BASE/"aptos2019"; FLAG=BASE/"flags"; CKPT=BASE/"checkpoints"
LOG=BASE/"logs"; CACHE=BASE/"cache"; ART=BASE/"artifacts"
PLOT=BASE/"plots"; EXPORT=BASE/"export"; DEPLOY=BASE/"deploy"
STATE_D=BASE/"state"; IMG_DIR=DATA/"train_images"; CSV_PATH=DATA/"train.csv"
for d in[DATA,FLAG,CKPT,LOG,CACHE,ART,PLOT,EXPORT,DEPLOY,STATE_D]:d.mkdir(parents=True,exist_ok=True)

def is_done(s):return(FLAG/f"{s}.done").exists()
def mark_done(s):(FLAG/f"{s}.done").touch()
def clear_done(s):
    f=FLAG/f"{s}.done"
    if f.exists():f.unlink()
def save_json(d,p):Path(p).write_text(json.dumps(d,indent=2,default=str))
def load_json(p):return json.loads(Path(p).read_text())
def save_pkl(o,p):
    with open(p,"wb")as f:pickle.dump(o,f)
def load_pkl(p):
    with open(p,"rb")as f:return pickle.load(f)

_TS=STATE_D/"train_state.pkl"
def ts_save(**kw):s=ts_load();s.update(kw);save_pkl(s,_TS)
def ts_load():
    if _TS.exists():return load_pkl(_TS)
    return{}
_MS=ART/"metrics_state.json"
def st_save(k,v):
    s={}
    if _MS.exists():
        try:s=json.loads(_MS.read_text())
        except:pass
    s[k]=v;_MS.write_text(json.dumps(s,indent=2,default=str))
def st_get(k,d=None):
    if _MS.exists():
        try:return json.loads(_MS.read_text()).get(k,d)
        except:pass
    return d

def prog(cur,tot,pre="",w=50,ex=""):
    p=cur/max(tot,1)*100;f=int(w*cur//max(tot,1))
    print(f"\r  {pre} [{'█'*f}{'░'*(w-f)}] {p:5.1f}% ({cur:,}/{tot:,}) {ex}",end="",flush=True)
    if cur>=tot:print()
def step_start(n,name):
    print("="*68);print(f"  [STEP {n:02d}] {name}");print("="*68);return time.time()
def step_skip(n,name,det=""):
    print("="*68);print(f"  [STEP {n:02d}] {name}");print("  ✅ COMPLETED → Skipping")
    if det:print(f"  {det}");print("="*68)
def step_end(n,t0,o=None):
    print(f"\n  ✅ Done in {time.time()-t0:.1f}s")
    if o:
        for k,v in o.items():print(f"  {k}: {v}")
    print("="*68)

NC=5;NF=5;SEED=42
GRADE={0:"No DR",1:"Mild",2:"Moderate",3:"Severe",4:"Proliferative"}
GCOL=["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]
IMEAN=[0.485,0.456,0.406];ISTD=[0.229,0.224,0.225]
THRESH=[0.7,1.5,2.5,3.5]
BACKBONES=[
    ("tf_efficientnet_b4",42),("tf_efficientnet_b4",123),
    ("tf_efficientnetv2_b1",42),("tf_efficientnetv2_b1",123),
    ("tf_efficientnet_b3",42),("tf_efficientnet_b3",123),
    ("seresnext50_32x4d",42),("seresnext50_32x4d",123),
]
done=sorted(FLAG.glob("*.done"))
print(f"  BASE: {BASE} | Models: {len(BACKBONES)} | Done: {len(done)}")
for f in done:print(f"    ✅ {f.stem}")
print("="*68)

  BASE: C:\Users\RAKSHITA BAI.J\DR_data | Models: 8 | Done: 20
    ✅ checkpoint
    ✅ cleaning
    ✅ data_load
    ✅ dataloader
    ✅ dataset
    ✅ download
    ✅ dr15_merge
    ✅ eda
    ✅ extract
    ✅ install
    ✅ kaggle_auth
    ✅ kfold
    ✅ label_analysis
    ✅ model
    ✅ preprocess
    ✅ preprocessing_fns
    ✅ split
    ✅ system
    ✅ training_config
    ✅ training_state


## Step 1 — System Setup (CUDA / MPS / CPU)


In [2]:
import random,numpy as np
if is_done("system"):
    info=load_json(LOG/"system_info.json");step_skip(1,"SYSTEM",f"Device={info.get('device')}")
else:
    t0=step_start(1,"SYSTEM SETUP")
    info={"os":f"{platform.system()} {platform.release()}","py":sys.version.split()[0]}
    try:
        import psutil;info["ram"]=round(psutil.virtual_memory().total/1e9,1)
    except:pass
    _,_,free=shutil.disk_usage(str(HOME));info["disk"]=round(free/1e9,1)
    # GPU
    gpu_hw=False
    try:
        r=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader"],capture_output=True,text=True,timeout=10)
        if r.returncode==0:
            gpu_hw=True
            for l in r.stdout.strip().split("\n"):print(f"    {l.strip()}")
            import re;m=re.search(r"CUDA Version:\s*([\d.]+)",subprocess.run(["nvidia-smi"],capture_output=True,text=True).stdout)
            if m:info["drv_cuda"]=m.group(1)
    except:print("    nvidia-smi not found")
    # PyTorch
    try:
        import torch;info["torch"]=torch.__version__;info["torch_cuda"]=str(torch.version.cuda)
        print(f"  PyTorch {torch.__version__} | CUDA built: {torch.version.cuda or 'NONE'}")
        if torch.cuda.is_available():
            info["device"]="cuda";info["gpu"]=torch.cuda.get_device_name(0)
            info["vram"]=round(torch.cuda.get_device_properties(0).total_mem/1e9,1)
            print(f"  ✅ {info['gpu']} ({info['vram']}GB)")
        elif gpu_hw:
            print("  ⚠️ GPU exists but PyTorch has NO CUDA! Step 2 will fix.");info["device"]="cpu"
        elif hasattr(torch.backends,"mps")and torch.backends.mps.is_available():
            info["device"]="mps";print("  ✅ MPS")
        else:info["device"]="cpu";print("  CPU only")
    except ImportError:info["device"]="pending";print("  PyTorch not installed")
    random.seed(SEED);np.random.seed(SEED)
    try:
        torch.manual_seed(SEED)
        if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED);torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    except:pass
    save_json(info,LOG/"system_info.json");mark_done("system");step_end(1,t0)

  [STEP 01] SYSTEM
  ✅ COMPLETED → Skipping
  Device=pending


## Step 2 — Install Requirements


In [3]:
import importlib
if is_done("install"):step_skip(2,"INSTALL")
else:
    t0=step_start(2,"INSTALL + CUDA FIX")
    need=False
    try:
        import torch
        if not torch.cuda.is_available():
            try:
                if subprocess.run(["nvidia-smi"],capture_output=True,timeout=5).returncode==0:need=True
            except:pass
    except ImportError:need=True
    if need and platform.system()in("Windows","Linux"):
        print("  📦 Installing PyTorch+CUDA12.4...")
        subprocess.run([sys.executable,"-m","pip","install","-q","torch","torchvision","torchaudio",
            "--index-url","https://download.pytorch.org/whl/cu124"],capture_output=True,text=True)
    elif need:
        subprocess.run([sys.executable,"-m","pip","install","-q","torch","torchvision","torchaudio"],capture_output=True,text=True)
    pkgs={"timm":"timm>=1.0.0","albumentations":"albumentations>=1.4.0","cv2":"opencv-python-headless",
        "sklearn":"scikit-learn","scipy":"scipy","pandas":"pandas","numpy":"numpy","tqdm":"tqdm",
        "matplotlib":"matplotlib","pytorch_grad_cam":"grad-cam","kaggle":"kaggle","pyarrow":"pyarrow",
        "PIL":"pillow<11.0","psutil":"psutil","ipywidgets":"ipywidgets"}
    miss=[p for m,p in pkgs.items() if not importlib.util.find_spec(m.split(".")[0])]
    if miss:
        for i,p in enumerate(miss):
            prog(i+1,len(miss),"Install",extra=p[:25])
            subprocess.run([sys.executable,"-m","pip","install","-q","--upgrade",p],capture_output=True,text=True)
    else:print("  ✅ All present")
    mark_done("install");step_end(2,t0)

  [STEP 02] INSTALL
  ✅ COMPLETED → Skipping


## Step 3 — Kaggle Authentication


In [4]:
if is_done("kaggle_auth"):step_skip(3,"KAGGLE AUTH")
else:
    t0=step_start(3,"KAGGLE AUTH")
    KD=HOME/".kaggle";KJ=KD/"kaggle.json"
    if KJ.exists():
        print(f"  ✅ User: {json.loads(KJ.read_text()).get('username')}");mark_done("kaggle_auth")
    else:
        ku=os.environ.get("KAGGLE_USERNAME","");kk=os.environ.get("KAGGLE_KEY","")
        if ku and kk:
            KD.mkdir(parents=True,exist_ok=True);KJ.write_text(json.dumps({"username":ku,"key":kk}))
            if platform.system()!="Windows":KJ.chmod(0o600)
            print(f"  ✅ From env: {ku}");mark_done("kaggle_auth")
        else:
            try:
                import ipywidgets as w;from IPython.display import display,HTML
                display(HTML("<h4>📁 Upload kaggle.json:</h4>"))
                up=w.FileUpload(accept=".json",multiple=False);lb=w.Label("⏳ Waiting...")
                def _on(c):
                    if up.value:
                        uv=list(up.value.values())[0] if isinstance(up.value,dict) else up.value[0]
                        ct=uv["content"] if isinstance(uv,dict) else uv.content
                        KD.mkdir(parents=True,exist_ok=True)
                        KJ.write_bytes(ct if isinstance(ct,bytes) else ct.tobytes())
                        if platform.system()!="Windows":KJ.chmod(0o600)
                        lb.value=f"✅ {json.loads(KJ.read_text()).get('username')}";mark_done("kaggle_auth")
                up.observe(_on,names="value");display(up,lb)
            except:print(f"  Place kaggle.json at: {KJ}")
    if KJ.exists():os.environ["KAGGLE_CONFIG_DIR"]=str(KD)
    step_end(3,t0)

  [STEP 03] KAGGLE AUTH
  ✅ COMPLETED → Skipping


## Step 4 — Dataset Download & Extraction


In [5]:
import zipfile
if is_done("extract"):
    n=len(list(IMG_DIR.glob("*.png")))if IMG_DIR.exists()else 0
    step_skip(4,"DOWNLOAD",f"{n:,} images")
else:
    t0=step_start(4,"DOWNLOAD & EXTRACT")
    if not(HOME/".kaggle"/"kaggle.json").exists():raise FileNotFoundError("Run Step 3")
    ZIP=DATA/"aptos2019-blindness-detection.zip"
    if not is_done("download"):
        if not ZIP.exists():
            print("  📥 Downloading...");r=subprocess.run([sys.executable,"-m","kaggle","competitions","download","-c","aptos2019-blindness-detection","-p",str(DATA)],capture_output=True,text=True)
            if r.returncode!=0:raise RuntimeError(f"Failed: {r.stderr[-300:]}")
        mark_done("download")
    if not ZIP.exists():
        zips=list(DATA.glob("*.zip"));ZIP=zips[0]if zips else None
        if not ZIP:raise FileNotFoundError("No ZIP")
    with zipfile.ZipFile(ZIP,"r")as zf:
        ms=zf.namelist();tot=len(ms)
        for i,m in enumerate(ms):
            zf.extract(m,DATA)
            if(i+1)%max(1,tot//25)==0 or i==tot-1:prog(i+1,tot,"Extract")
    mark_done("extract");step_end(4,t0)

  [STEP 04] DOWNLOAD
  ✅ COMPLETED → Skipping
  3,662 images


## Step 4b — DR 2015 Download & Merge (Stage 1 data strategy)


In [6]:
# ── Step 4b — DR 2015 Download & Merge ──────────────────────────────────────
import zipfile, time
import pandas as pd
import numpy as np
from pathlib import Path

DR15_DIR   = BASE / "dr2015"
DR15_TRAIN = DR15_DIR / "train"
DR15_TEST  = DR15_DIR / "test"
for d in [DR15_DIR, DR15_TRAIN, DR15_TEST]:
    d.mkdir(parents=True, exist_ok=True)

# ── Safe step helpers that accept string step IDs like "4b" ─────────────────
def _fmt_step(n):
    return f"{n:02d}" if isinstance(n, int) else str(n)

def step_start_s(n, name):
    print("="*68)
    print(f"  [STEP {_fmt_step(n)}] {name}")
    print("="*68)
    return time.time()

def step_skip_s(n, name, det=""):
    print("="*68)
    print(f"  [STEP {_fmt_step(n)}] {name}")
    print("  ✅ COMPLETED → Skipping")
    if det: print(f"  {det}")
    print("="*68)

def step_end_s(n, t0, o=None):
    print(f"\n  ✅ Done in {time.time()-t0:.1f}s")
    if o:
        for k, v in o.items(): print(f"  {k}: {v}")
    print("="*68)

# ── Kaggle download helper (fixes "cannot be directly executed") ─────────────
def kaggle_download(cmd_args):
    """
    Try multiple kaggle invocation methods in order:
      1. kaggle CLI via entry_points (most reliable)
      2. python -m kaggle.cli
      3. python -c import kaggle; kaggle.cli.main()
    Returns (success:bool, stderr:str)
    """
    import shutil
    # Method 1: direct executable on PATH
    kaggle_exe = shutil.which("kaggle")
    if kaggle_exe:
        r = subprocess.run([kaggle_exe] + cmd_args,
                           capture_output=True, text=True)
        if r.returncode == 0:
            return True, ""
        # fall through to next method
    # Method 2: python -m kaggle.cli
    r = subprocess.run(
        [sys.executable, "-m", "kaggle.cli"] + cmd_args,
        capture_output=True, text=True)
    if r.returncode == 0:
        return True, ""
    # Method 3: inline import
    r2 = subprocess.run(
        [sys.executable, "-c",
         f"from kaggle.cli import main; import sys; sys.argv=['kaggle']+{cmd_args!r}; main()"],
        capture_output=True, text=True)
    if r2.returncode == 0:
        return True, ""
    return False, r2.stderr

if is_done("dr15_merge"):
    step_skip_s("4b", "DR 2015 DOWNLOAD & MERGE")
else:
    t0 = step_start_s("4b", "DR 2015 DOWNLOAD & MERGE")

    # ── 1. Download ──────────────────────────────────────────────────────────
    DR15_ZIP = DR15_DIR / "diabetic-retinopathy-detection.zip"
    if not is_done("dr15_download"):
        if not DR15_ZIP.exists():
            print("  📥 Downloading DR 2015 (~35 GB, may take a while)...")
            ok, err = kaggle_download([
                "competitions", "download",
                "-c", "diabetic-retinopathy-detection",
                "-p", str(DR15_DIR)
            ])
            if not ok:
                print(f"  ⚠️  DR 2015 download failed:\n  {err[-400:]}")
                print("  ↳  Continuing with APTOS 2019 only.")
                # Graceful fallback — build APTOS-only merged df
                if (ART / "df_clean.parquet").exists():
                    df_merged = pd.read_parquet(ART / "df_clean.parquet")
                else:
                    df_merged = pd.read_csv(CSV_PATH)
                df_merged["image_path"] = df_merged["id_code"].apply(
                    lambda x: str(IMG_DIR / f"{x}.png"))
                df_merged["source"] = "aptos2019"
                df_merged.to_parquet(ART / "df_merged.parquet", index=False)
                print(f"  APTOS 2019 only: {len(df_merged):,} images")
                mark_done("dr15_merge")
                step_end_s("4b", t0)
            else:
                mark_done("dr15_download")
                print("  ✅ DR 2015 downloaded")
        else:
            mark_done("dr15_download")
            print("  ✅ DR 2015 zip already present")

    # ── 2. Extract ───────────────────────────────────────────────────────────
    if not is_done("dr15_extract") and is_done("dr15_download"):
        zips = list(DR15_DIR.glob("*.zip"))
        if not zips:
            zips = list(DR15_DIR.rglob("*.zip"))
        for zp in zips:
            try:
                with zipfile.ZipFile(zp, "r") as zf:
                    members  = zf.namelist()
                    csv_mbrs = [m for m in members if m.endswith(".csv")]
                    img_mbrs = [m for m in members
                                if m.lower().endswith((".jpeg", ".jpg", ".png"))]
                    tot    = len(csv_mbrs) + len(img_mbrs)
                    done_n = 0
                    for m in csv_mbrs:
                        zf.extract(m, DR15_DIR)
                        done_n += 1
                    for i, m in enumerate(img_mbrs):
                        zf.extract(m, DR15_DIR)
                        done_n += 1
                        if (i + 1) % max(1, len(img_mbrs) // 20) == 0 \
                                or i == len(img_mbrs) - 1:
                            prog(done_n, tot, "Extract DR15")
                print(f"\n  ✅ Extracted: {zp.name}")
            except Exception as e:
                print(f"  ⚠️  Extract error [{zp.name}]: {e}")
        mark_done("dr15_extract")

    # ── 3. Load labels ───────────────────────────────────────────────────────
    def _load_dr15_csv(path):
        tmp = pd.read_csv(path)
        col_map = {}
        for c in tmp.columns:
            cl = c.lower()
            if cl in ("image", "image_name", "id_code", "filename"):
                col_map[c] = "id_code"
            elif cl in ("level", "diagnosis", "label", "grade"):
                col_map[c] = "diagnosis"
        tmp = tmp.rename(columns=col_map)
        if "id_code" not in tmp.columns:
            tmp.columns = ["id_code", "diagnosis"] + list(tmp.columns[2:])
        tmp = tmp[["id_code", "diagnosis"]].copy()
        tmp["id_code"]   = tmp["id_code"].astype(str).str.strip()
        tmp["diagnosis"] = pd.to_numeric(tmp["diagnosis"], errors="coerce")
        tmp = tmp.dropna(subset=["diagnosis"])
        tmp["diagnosis"] = tmp["diagnosis"].astype(int).clip(0, 4)
        return tmp

    df_dr15 = None
    for csv_name in ("trainLabels.csv", "train_labels.csv", "labels.csv"):
        hits = list(DR15_DIR.rglob(csv_name))
        if hits:
            df_dr15 = _load_dr15_csv(hits[0])
            print(f"  DR 2015 train labels : {len(df_dr15):,}  ({hits[0].name})")
            break

    for csv_name in ("testLabels.csv", "test_labels.csv"):
        hits = list(DR15_DIR.rglob(csv_name))
        if hits:
            df_dr15_test = _load_dr15_csv(hits[0])
            print(f"  DR 2015 test  labels : {len(df_dr15_test):,}  ({hits[0].name})")
            df_dr15 = pd.concat([df_dr15, df_dr15_test], ignore_index=True) \
                      if df_dr15 is not None else df_dr15_test
            break

    # ── 4. Map image paths ───────────────────────────────────────────────────
    if df_dr15 is not None and len(df_dr15) > 0:
        img_exts = {".jpeg", ".jpg", ".png"}
        print("  🔎 Scanning DR 2015 images...")
        dr15_img_map = {}
        for p in DR15_DIR.rglob("*"):
            if p.suffix.lower() in img_exts:
                dr15_img_map[p.stem] = str(p)
        print(f"  Found {len(dr15_img_map):,} DR 2015 image files on disk")

        df_dr15["image_path"] = df_dr15["id_code"].map(dr15_img_map)
        before = len(df_dr15)
        df_dr15 = df_dr15[df_dr15["image_path"].notna()].reset_index(drop=True)
        print(f"  DR 2015 matched : {len(df_dr15):,} / {before:,}")

        # ── 5. Merge APTOS 2019 + DR 2015 ───────────────────────────────────
        if (ART / "kfold_splits.parquet").exists():
            df_aptos = pd.read_parquet(ART / "kfold_splits.parquet")
        elif (ART / "df_clean.parquet").exists():
            df_aptos = pd.read_parquet(ART / "df_clean.parquet")
        else:
            df_aptos = pd.read_csv(CSV_PATH)

        df_aptos["image_path"] = df_aptos["id_code"].apply(
            lambda x: str(IMG_DIR / f"{x}.png"))
        df_aptos["source"] = "aptos2019"
        df_dr15["source"]  = "dr2015"

        keep_cols = ["id_code", "diagnosis", "image_path", "source"]
        for col in keep_cols:
            if col not in df_aptos.columns: df_aptos[col] = ""
            if col not in df_dr15.columns:  df_dr15[col]  = ""

        df_merged = pd.concat(
            [df_aptos[keep_cols], df_dr15[keep_cols]], ignore_index=True)
        df_merged["diagnosis"] = df_merged["diagnosis"].astype(int).clip(0, 4)
        df_merged = df_merged[
            df_merged["image_path"].apply(lambda p: Path(p).exists())
        ].reset_index(drop=True)

        print(f"\n  ✅ Merged dataset  : {len(df_merged):,} images total")
        print(f"     APTOS 2019     : {(df_merged['source']=='aptos2019').sum():,}")
        print(f"     DR 2015        : {(df_merged['source']=='dr2015').sum():,}")
        for g in range(5):
            print(f"     G{g}             : {(df_merged['diagnosis']==g).sum():,}")

        df_merged.to_parquet(ART / "df_merged.parquet", index=False)
        save_json({
            "aptos2019": int((df_merged["source"] == "aptos2019").sum()),
            "dr2015":    int((df_merged["source"] == "dr2015").sum()),
            "total":     len(df_merged)
        }, LOG / "merge_info.json")

    else:
        # Graceful fallback — APTOS 2019 only
        print("  ⚠️  DR 2015 labels not found — using APTOS 2019 only")
        if (ART / "df_clean.parquet").exists():
            df_merged = pd.read_parquet(ART / "df_clean.parquet")
        else:
            df_merged = pd.read_csv(CSV_PATH)
        df_merged["image_path"] = df_merged["id_code"].apply(
            lambda x: str(IMG_DIR / f"{x}.png"))
        df_merged["source"] = "aptos2019"
        df_merged.to_parquet(ART / "df_merged.parquet", index=False)
        print(f"  APTOS 2019 only: {len(df_merged):,} images")

    mark_done("dr15_merge")
    step_end_s("4b", t0)

  [STEP 4b] DR 2015 DOWNLOAD & MERGE
  ✅ COMPLETED → Skipping


## Step 5 — Load Dataset


In [7]:
# ── Step 5 — Load Dataset ────────────────────────────────────────────────────
import os, sys, json, gc, time, random, shutil, warnings, pickle, platform, re, subprocess, importlib
from pathlib import Path
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Auto-install any missing core packages before importing ──────────────────
def _pip(pkg, import_as=None):
    name = import_as or pkg.split(">=")[0].split("==")[0].replace("-","_")
    if importlib.util.find_spec(name.split(".")[0]) is None:
        print(f"  📦 Installing {pkg}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
                       capture_output=True)

# PyTorch — detect GPU and install correct wheel
def _ensure_torch():
    if importlib.util.find_spec("torch") is not None:
        return
    print("  📦 Installing PyTorch...")
    try:
        r = subprocess.run(["nvidia-smi"], capture_output=True, timeout=5)
        has_gpu = r.returncode == 0
    except Exception:
        has_gpu = False
    if has_gpu and platform.system() in ("Windows", "Linux"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "torch", "torchvision", "torchaudio",
                        "--index-url", "https://download.pytorch.org/whl/cu124"],
                       capture_output=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "torch", "torchvision", "torchaudio"],
                       capture_output=True)
    print("  ✅ PyTorch installed — re-importing...")

_ensure_torch()
_pip("timm>=1.0.0",         "timm")
_pip("albumentations>=1.4.0","albumentations")
_pip("opencv-python-headless","cv2")
_pip("scikit-learn",         "sklearn")
_pip("scipy",                "scipy")
_pip("tqdm",                 "tqdm")
_pip("matplotlib",           "matplotlib")
_pip("Pillow",               "PIL")
_pip("pyarrow",              "pyarrow")

# ── Now safe to import everything ────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
from scipy.optimize import minimize
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, cohen_kappa_score,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score
)
warnings.filterwarnings("ignore")

t0 = step_start(5, "LOAD DATASET")

# ── Seeding & device ─────────────────────────────────────────────────────────
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

seed_everything()

if torch.cuda.is_available():
    DEV  = "cuda"; AMP  = True;  PINM = True
    print(f"  🔥 {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEV  = "mps";  AMP  = False; PINM = False
    print("  🍎 MPS")
else:
    DEV  = "cpu";  AMP  = False; PINM = False
    print("  💻 CPU")

NW = 0 if platform.system() in ("Darwin", "Windows") \
       else min(4, os.cpu_count() or 1)

def safe_load(p, ml="cpu"):
    try:
        return torch.load(p, map_location=ml, weights_only=False)
    except TypeError:
        return torch.load(p, map_location=ml)

# ── Load CSV ──────────────────────────────────────────────────────────────────
if not CSV_PATH.exists():
    raise FileNotFoundError(f"train.csv not found at {CSV_PATH} — run Step 4 first")

df = pd.read_csv(CSV_PATH)
df["image_path"]  = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
df["grade_label"] = df["diagnosis"].map(GRADE)
df["binary"]      = (df["diagnosis"] >= 1).astype(int)

# ── Apply cleaning overlay if available ──────────────────────────────────────
cp = ART / "df_clean.parquet"
if is_done("cleaning") and cp.exists():
    df = pd.read_parquet(cp)
    df["image_path"]  = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
    df["grade_label"] = df["diagnosis"].map(GRADE)
    df["binary"]      = (df["diagnosis"] >= 1).astype(int)

# ── Apply kfold splits overlay if available ───────────────────────────────────
sp = ART / "kfold_splits.parquet"
if is_done("kfold") and sp.exists():
    df = pd.read_parquet(sp)
    df["image_path"]  = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
    df["grade_label"] = df["diagnosis"].map(GRADE)
    df["binary"]      = (df["diagnosis"] >= 1).astype(int)

# ── Apply merged (APTOS+DR2015) overlay if available ─────────────────────────
mp = ART / "df_merged.parquet"
if mp.exists():
    df_merged = pd.read_parquet(mp)
    # Fix image_path per source
    def _fix_path(row):
        src = row.get("source", "aptos2019")
        if src == "aptos2019":
            return str(IMG_DIR / f'{row["id_code"]}.png')
        return str(row["image_path"])  # DR2015 / IDRiD / Messidor keep their path
    df_merged["image_path"]  = df_merged.apply(_fix_path, axis=1)
    df_merged["grade_label"] = df_merged["diagnosis"].map(GRADE)
    df_merged["binary"]      = (df_merged["diagnosis"] >= 1).astype(int)
    # If kfold splits exist, restore fold column for APTOS rows
    if "fold" in df.columns and "fold" not in df_merged.columns:
        fold_map = df.set_index("id_code")["fold"].to_dict()
        df_merged["fold"] = df_merged["id_code"].map(fold_map).fillna(-1).astype(int)
    df = df_merged.copy()
    print(f"  Using MERGED dataset (APTOS+DR2015): {len(df):,} images")

# ── Cache flag ────────────────────────────────────────────────────────────────
USE_CACHE = is_done("preprocess") and CACHE.exists()
CACHE_SZ  = 384

# ── Drop missing images ───────────────────────────────────────────────────────
exists_mask = df["image_path"].apply(lambda p: Path(p).exists())
miss_count  = (~exists_mask).sum()
if miss_count:
    print(f"  ⚠️  Dropping {miss_count} rows with missing image files")
    df = df[exists_mask].reset_index(drop=True)

print(f"  {len(df):,} images | {NC} classes | Cache={USE_CACHE} | Device={DEV}")
for g in range(5):
    print(f"    G{g} ({GRADE[g]}): {(df['diagnosis']==g).sum():,}")

mark_done("data_load")
step_end(5, t0)

  📦 Installing PyTorch...
  ✅ PyTorch installed — re-importing...


ModuleNotFoundError: No module named 'torch'

## Step 4c — Apply Merged Dataset to df


In [ ]:
# Apply merged APTOS+DR2015 dataset if available
if (ART/"df_merged.parquet").exists():
    df_merged = pd.read_parquet(ART/"df_merged.parquet")
    df_merged["image_path"] = df_merged.apply(
        lambda r: str(IMG_DIR/f'{r["id_code"]}.png') if r.get("source","aptos2019")=="aptos2019"
                  else str(r["image_path"]), axis=1)
    df_merged = df_merged[df_merged["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
    df_merged["diagnosis"] = df_merged["diagnosis"].astype(int).clip(0,4)
    df_merged["grade_label"] = df_merged["diagnosis"].map(GRADE)
    df_merged["binary"] = (df_merged["diagnosis"]>=1).astype(int)
    df = df_merged.copy()
    print(f"  ✅ Using MERGED dataset: {len(df):,} images (APTOS+DR2015)")
else:
    print(f"  Using APTOS 2019 only: {len(df):,} images")
for g in range(5):
    print(f"    G{g}: {(df['diagnosis']==g).sum():,}")


## Step 6 — Data Cleaning


In [ ]:
if is_done("cleaning"):
    df=pd.read_parquet(ART/"df_clean.parquet");df["image_path"]=df["id_code"].apply(lambda x:str(IMG_DIR/f"{x}.png"))
    df["grade_label"]=df["diagnosis"].map(GRADE);df["binary"]=(df["diagnosis"]>=1).astype(int)
    step_skip(6,"CLEANING",f"{len(df):,}")
else:
    t0=step_start(6,"DATA CLEANING (minimal)")
    tot=len(df);bad=[]
    for i,(_,row)in enumerate(df.iterrows()):
        bgr=cv2.imread(str(row["image_path"]));r=None
        if bgr is None:r="unreadable"
        elif bgr.shape[0]<50 or bgr.shape[1]<50:r="tiny"
        elif float(cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY).mean())<3:r="black"
        if r:bad.append((row["id_code"],r))
        if(i+1)%max(1,tot//20)==0 or i==tot-1:prog(i+1,tot,"Check")
    bids=set(x[0]for x in bad);df=df[~df["id_code"].isin(bids)].reset_index(drop=True)
    df.to_parquet(ART/"df_clean.parquet",index=False)
    print(f"\n  Removed: {len(bad)}");mark_done("cleaning");step_end(6,t0)

## Step 7 — EDA


In [ ]:
if is_done("eda"):step_skip(7,"EDA")
else:
    t0=step_start(7,"EDA")
    try:get_ipython().run_line_magic('matplotlib','inline')
    except:pass
    fig,ax=plt.subplots(1,2,figsize=(14,5))
    vc=df["diagnosis"].value_counts().sort_index()
    ax[0].bar([GRADE[i]for i in range(5)],vc.values,color=GCOL,edgecolor="k")
    ax[0].set_title("Distribution",fontweight="bold")
    for i,v in enumerate(vc.values):ax[0].text(i,v+20,str(v),ha="center")
    ax[1].pie(vc.values,labels=[GRADE[i]for i in range(5)],colors=GCOL,autopct="%1.1f%%")
    plt.suptitle(f"APTOS — {len(df):,}",fontweight="bold")
    plt.tight_layout();plt.savefig(PLOT/"eda.png",dpi=150);plt.show()
    mark_done("eda");step_end(7,t0)

## Step 8 — Label Analysis


In [ ]:
if is_done("label_analysis"):step_skip(8,"LABELS")
else:
    t0=step_start(8,"LABEL ANALYSIS")
    vc=df["diagnosis"].value_counts().sort_index()
    c=np.bincount(df["diagnosis"].values,minlength=5).astype(float)
    w=len(df)/(5*np.maximum(c,1));w=w/w.sum()*5
    for g,cnt in vc.items():print(f"  G{g}: {cnt:5d} w={w[g]:.3f}")
    save_json({"w":{str(i):round(float(w[i]),4)for i in range(5)}},LOG/"weights.json")
    mark_done("label_analysis");step_end(8,t0)

## Step 9 — Preprocessing (WINNER: resize + normalize ONLY)


In [ ]:
t0=step_start(9,"PREPROCESSING")
IMG_SIZE=int(os.environ.get("IMG_SIZE",512))
def preprocess_fundus(path_or_arr,size=None):
    target=size or IMG_SIZE
    if isinstance(path_or_arr,np.ndarray):rgb=path_or_arr.copy()
    else:
        bgr=cv2.imread(str(path_or_arr))
        if bgr is None:return np.zeros((target,target,3),np.uint8)
        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    gray=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY)
    _,th=cv2.threshold(gray,7,255,cv2.THRESH_BINARY)
    co=cv2.findNonZero(th)
    if co is not None:x,y,w,h=cv2.boundingRect(co);rgb=rgb[y:y+h,x:x+w]
    return cv2.resize(rgb,(target,target),interpolation=cv2.INTER_AREA)
print(f"  Resize {IMG_SIZE}px | NO CLAHE | NO enhancement")
mark_done("preprocessing");step_end(9,t0)

## Step 10 — Preprocessing Cache


In [ ]:
if is_done("preprocess"):
    USE_CACHE=True;step_skip(10,"CACHE",f"{len(list(CACHE.glob('*.npy'))):,}")
else:
    t0=step_start(10,"CACHE")
    ex=set(p.stem for p in CACHE.glob("*.npy"));todo=df[~df["id_code"].isin(ex)];ok=0
    for i,(_,row)in enumerate(todo.iterrows()):
        try:np.save(str(CACHE/f'{row["id_code"]}.npy'),preprocess_fundus(row["image_path"],CACHE_SZ));ok+=1
        except:pass
        if(i+1)%max(1,len(todo)//20)==0 or i==len(todo)-1:prog(i+1,len(todo),"Cache")
    USE_CACHE=True;mark_done("preprocess");step_end(10,t0)

## Step 11 — Train / Test Split


In [ ]:
if is_done("split"):step_skip(11,"SPLIT")
else:
    t0=step_start(11,"TRAIN/TEST SPLIT")
    print("  Fold 0 = hold-out test set (never trained on)")
    print("  Folds 1-4 = cross-validation")
    mark_done("split");step_end(11,t0)

## Step 12 — Stratified K-Fold (5)


In [ ]:
if is_done("kfold"):
    df=pd.read_parquet(ART/"kfold_splits.parquet");df["image_path"]=df["id_code"].apply(lambda x:str(IMG_DIR/f"{x}.png"))
    df["grade_label"]=df["diagnosis"].map(GRADE);df["binary"]=(df["diagnosis"]>=1).astype(int)
    step_skip(12,"K-FOLD",f"{len(df):,}")
else:
    t0=step_start(12,"K-FOLD (5 folds)")
    skf=StratifiedKFold(n_splits=NF,shuffle=True,random_state=SEED)
    df["fold"]=-1
    for fi,(_,vi)in enumerate(skf.split(df,df["diagnosis"])):df.loc[vi,"fold"]=fi
    df.to_parquet(ART/"kfold_splits.parquet",index=False)
    for f in range(NF):
        gd=df[df["fold"]==f]["diagnosis"].value_counts().sort_index()
        print(f"  Fold {f}: {(df['fold']==f).sum()} [{' '.join(f'G{g}:{c}' for g,c in gd.items())}]")
    mark_done("kfold");step_end(12,t0)

## Step 13 — Augmentation + Dataset Pipeline


In [ ]:
t0=step_start(13,"AUGMENTATION + DATASET")

def build_train_tf(sz):
    return A.Compose([
        A.RandomResizedCrop(size=(sz,sz),scale=(0.8,1.0),ratio=(0.9,1.1)),
        A.HorizontalFlip(p=0.5),A.VerticalFlip(p=0.3),
        A.Rotate(limit=180,p=0.7),
        A.ShiftScaleRotate(shift_limit=0.1,scale_limit=0.15,rotate_limit=45,p=0.5),
        A.RandomBrightnessContrast(0.2,0.2,p=0.5),
        A.HueSaturationValue(10,20,10,p=0.3),
        A.OneOf([A.GaussianBlur(blur_limit=(3,5)),A.Sharpen()],p=0.3),
        A.CoarseDropout(num_holes_range=(1,8),hole_height_range=(sz//20,sz//10),
                        hole_width_range=(sz//20,sz//10),p=0.2),
        A.Normalize(mean=IMEAN,std=ISTD),ToTensorV2()])

def build_val_tf(sz):
    return A.Compose([A.Resize(sz,sz),A.Normalize(mean=IMEAN,std=ISTD),ToTensorV2()])

def build_tta_tf(sz):
    n=A.Normalize(mean=IMEAN,std=ISTD);r=A.Resize(sz,sz)
    return[build_val_tf(sz),
        A.Compose([A.HorizontalFlip(p=1),r,n,ToTensorV2()]),
        A.Compose([A.Rotate(limit=(10,10),p=1),r,n,ToTensorV2()]),
        A.Compose([A.Rotate(limit=(-10,-10),p=1),r,n,ToTensorV2()]),
        A.Compose([A.VerticalFlip(p=1),r,n,ToTensorV2()])]

class DRDataset(Dataset):
    def __init__(self,df,tf=None,sz=None,use_cache=True):
        self.df=df.reset_index(drop=True);self.tf=tf;self.sz=sz or IMG_SIZE;self.uc=use_cache and USE_CACHE
    def __len__(self):return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx];lab=float(row["diagnosis"])
        if self.uc:
            cp=CACHE/f'{row["id_code"]}.npy'
            if cp.exists():
                img=np.load(str(cp))
                if self.sz!=CACHE_SZ:img=cv2.resize(img,(self.sz,self.sz))
            else:img=preprocess_fundus(row["image_path"],self.sz)
        else:img=preprocess_fundus(row["image_path"],self.sz)
        if self.tf:img=self.tf(image=img)["image"]
        else:img=torch.from_numpy(img.transpose(2,0,1)).float()/255.0
        return img,torch.tensor(lab,dtype=torch.float32)

def build_sampler(df_s):
    labs=df_s["diagnosis"].values;c=np.bincount(labs,minlength=5).astype(float)
    w=1.0/np.maximum(c,1);sw=w[labs]
    return WeightedRandomSampler(torch.DoubleTensor(sw),len(sw),replacement=True)

def make_loader(ds,bs=16,shuffle=True,sampler=None,drop_last=False):
    if sampler:shuffle=False
    return DataLoader(ds,batch_size=bs,shuffle=shuffle,sampler=sampler,
        num_workers=NW,pin_memory=PINM,persistent_workers=(NW>0),drop_last=drop_last)

BS={224:16,384:8,512:2}
def qwk(yt,yp):return cohen_kappa_score(yt,yp,weights="quadratic")
def acc_fn(yt,yp):return(np.array(yt)==np.array(yp)).mean()
def reg2cls(p,t=THRESH):return np.digitize(np.clip(np.array(p),0,4),t).astype(int)

mark_done("dataset");print(f"  Regression | SmoothL1 | Thresholds: {THRESH}");step_end(13,t0)

## Step 14 — DataLoader Creation


In [ ]:
if is_done("dataloader"):step_skip(14,"DATALOADER")
else:
    t0=step_start(14,"DATALOADER")
    d=DRDataset(df[df["fold"]!=0].head(32),build_val_tf(224),224)
    l=make_loader(d,bs=8);im,lb=next(iter(l))
    print(f"  Shape: {im.shape} | Labels (float): {lb[:4].tolist()}")
    del d,l;gc.collect();mark_done("dataloader");step_end(14,t0)

## Step 15 — Model Architecture (GeM + Regression)


In [ ]:
t0=step_start(15,"MODEL")
class GeM(nn.Module):
    def __init__(self,p=3,eps=1e-6):
        super().__init__();self.p=nn.Parameter(torch.ones(1)*p);self.eps=eps
    def forward(self,x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),(x.size(-2),x.size(-1))).pow(1./self.p)

class DRModel(nn.Module):
    def __init__(self,backbone="tf_efficientnetv2_b1",drop=0.5,pretrained=True):
        super().__init__()
        self.backbone=timm.create_model(backbone,pretrained=pretrained,num_classes=0,global_pool="")
        fd=self.backbone.num_features;self.pool=GeM()
        self.head=nn.Sequential(nn.Flatten(),nn.Linear(fd,256),nn.BatchNorm1d(256),
            nn.ReLU(True),nn.Dropout(drop),nn.Linear(256,1))
    def forward(self,x):return self.head(self.pool(self.backbone(x))).squeeze(-1)
    def freeze_backbone(self):
        for p in self.backbone.parameters():p.requires_grad_(False)
    def unfreeze_top(self,n=4):
        for p in self.backbone.parameters():p.requires_grad_(False)
        children=list(self.backbone.children())
        for b in children[-n:]:
            for p in b.parameters():p.requires_grad_(True)
    def unfreeze_all(self):
        for p in self.parameters():p.requires_grad_(True)

def build_model(bb="tf_efficientnetv2_b1",pretrained=True):
    return DRModel(bb,0.5,pretrained).to(DEV)

class EMA:
    def __init__(self,model,decay=0.9999):
        self.decay=decay;self.shadow={n:p.clone().detach()for n,p in model.named_parameters()if p.requires_grad}
    def update(self,model):
        for n,p in model.named_parameters():
            if p.requires_grad and n in self.shadow:self.shadow[n].mul_(self.decay).add_(p.data,alpha=1-self.decay)
    def apply(self,model):
        self.backup={n:p.clone()for n,p in model.named_parameters()if n in self.shadow}
        for n,p in model.named_parameters():
            if n in self.shadow:p.data.copy_(self.shadow[n])
    def restore(self,model):
        for n,p in model.named_parameters():
            if n in self.backup:p.data.copy_(self.backup[n])

print(f"  Backbones: {[b[0]for b in BACKBONES[:4]]} (×2 seeds)")
print(f"  Head: GeM→Linear(256)→BN→ReLU→Drop(0.5)→Linear(1)")
print(f"  EMA: 0.9999");mark_done("model");step_end(15,t0)

## Step 16 — Training Config (SmoothL1 + AdamW + Warmup+Cosine)


In [ ]:
t0=step_start(16,"TRAINING CONFIG")
print("  Loss     : SmoothL1Loss ONLY")
print("  Optimizer: AdamW (lr=1e-4, wd=1e-4)")
print("  Scheduler: Warmup(5ep) + CosineAnnealing")
print("  AMP      : CUDA=ON, MPS=OFF, CPU=OFF")
print("  Grad clip: 1.0")
mark_done("training_config");step_end(16,t0)

## Step 17 — Checkpoint System


In [ ]:
t0=step_start(17,"CHECKPOINT SYSTEM")
print("  Per epoch saves: model, EMA, optimizer, scheduler, scaler, epoch, batch, best_qwk")
print("  Resume: fold → phase → epoch → batch")
print(f"  State file: {_TS}")
mark_done("checkpoint");step_end(17,t0)

## Step 18 — Training State Management


In [ ]:
t0=step_start(18,"TRAINING STATE")
st_save("version","v23")
print("  Tracks: loss, QWK, acc per epoch/fold")
print("  Best model saved using EMA weights")
print("  Exact resume from any crash point")
mark_done("training_state");step_end(18,t0)

## Step 19 — Training (3-Phase × 5-Fold)
- **Phase 1:** 224px, 30ep, frozen backbone, bs=16
- **Phase 2:** 384px, 80ep, partial unfreeze, bs=8, accum=2
- **Phase 3:** 512px, 50ep, full unfreeze, bs=2, accum=4


In [ ]:
WD=1e-4
PHASES=[
    {"sz":224,"ep":30,"name":"P1-Freeze","unf":0,"lr":1e-4,"acc":1},
    {"sz":384,"ep":80,"name":"P2-Partial","unf":4,"lr":5e-5,"acc":2},
    {"sz":512,"ep":50,"name":"P3-Full","unf":99,"lr":1e-5,"acc":4},
]
ES_PAT=10;ES_D=0.0005
CUR_BB=BACKBONES[0][0]

if is_done("training"):
    oof_preds=np.load(str(ART/"oof_preds.npy"));oof_labels=np.load(str(ART/"oof_labels.npy"))
    fold_qwks=load_json(ART/"fold_qwks.json")
    step_skip(19,"TRAINING",f"Mean QWK={np.mean(fold_qwks):.4f}")
else:
    t0=step_start(19,"TRAINING")
    oof_preds=np.zeros(len(df),dtype=np.float32);oof_labels=df["diagnosis"].values.copy()
    fold_qwks=[];crit=nn.SmoothL1Loss()

    for fold in range(NF):
        fc=CKPT/f"fold{fold}_best.pt";fo=ART/f"fold{fold}_oof.npy";ff=f"fold{fold}"
        if is_done(ff)and fc.exists():
            if fo.exists():oof_preds[df[df["fold"]==fold].index]=np.load(str(fo))
            prev=safe_load(fc,"cpu");fold_qwks.append(prev.get("val_qwk",0.0))
            print(f"  ✅ [RESUME] Fold {fold} QWK={fold_qwks[-1]:.4f}");continue

        print(f"\n  {'═'*20} FOLD {fold} {'═'*20}")
        seed_everything(SEED+fold)
        df_tr=df[df["fold"]!=fold].reset_index(drop=True)
        df_va=df[df["fold"]==fold].reset_index(drop=True)
        vi=df[df["fold"]==fold].index
        model=build_model(CUR_BB,True);ema=EMA(model);best_qwk=-1.0;best_state=None

        for pi,ph in enumerate(PHASES):
            sz=ph["sz"];nep=ph["ep"];pn=ph["name"];unf=ph["unf"];plr=ph["lr"];accum=ph["acc"]
            bs=BS[sz];pc=CKPT/f"fold{fold}_p{pi}.pt"
            if unf==0:model.freeze_backbone()
            elif unf>=99:model.unfreeze_all()
            else:model.unfreeze_top(unf)
            tp=sum(p.numel()for p in model.parameters()if p.requires_grad)
            print(f"  [{pn}] {sz}px ×{nep}ep bs={bs} acc={accum} | {tp/1e6:.2f}M")

            tr_ds=DRDataset(df_tr,build_train_tf(sz),sz);va_ds=DRDataset(df_va,build_val_tf(sz),sz)
            smp=build_sampler(df_tr)
            tr_ld=make_loader(tr_ds,bs,sampler=smp,drop_last=True)
            va_ld=make_loader(va_ds,bs,shuffle=False)

            opt=torch.optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=plr,weight_decay=WD)
            _w=5;_n=nep
            def _lr(ep,warmup=_w,total=_n):
                if ep<warmup:return(ep+1)/warmup
                return 0.5*(1+np.cos(np.pi*(ep-warmup)/(total-warmup)))
            sched=torch.optim.lr_scheduler.LambdaLR(opt,_lr)
            scaler=torch.amp.GradScaler("cuda")if AMP else None

            ep_start=0;batch_start=0
            if pc.exists():
                ps=safe_load(pc,DEV);model.load_state_dict(ps["model_state"])
                opt.load_state_dict(ps["optimizer_state"]);sched.load_state_dict(ps["scheduler_state"])
                ep_start=ps["epoch"];best_qwk=ps.get("best_qwk",best_qwk)
                if ps.get("best_state"):best_state=ps["best_state"]
                if ps.get("ema"):ema.shadow=ps["ema"]
                batch_start=ps.get("batch",0)
                if scaler and ps.get("scaler"):scaler.load_state_dict(ps["scaler"])
                print(f"    ↻ Resume ep={ep_start+1} batch={batch_start}")

            model.to(DEV);es=0
            for ep in range(ep_start,nep):
                model.train();el=0.0;nb=0;opt.zero_grad()
                for step,(imgs,labs)in enumerate(tr_ld):
                    if ep==ep_start and step<batch_start:continue
                    imgs,labs=imgs.to(DEV),labs.to(DEV)
                    if AMP:
                        with torch.amp.autocast("cuda"):out=model(imgs);loss=crit(out,labs)/accum
                        scaler.scale(loss).backward()
                        if(step+1)%accum==0:
                            scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                            scaler.step(opt);scaler.update();opt.zero_grad()
                    else:
                        out=model(imgs);loss=crit(out,labs)/accum;loss.backward()
                        if(step+1)%accum==0:
                            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step();opt.zero_grad()
                    el+=loss.item()*accum;nb+=1;ema.update(model)
                batch_start=0;sched.step();tl=el/max(nb,1)

                ema.apply(model);model.eval();vp=[];vl=[]
                with torch.no_grad():
                    for imgs,labs in va_ld:
                        out=model(imgs.to(DEV)).cpu().numpy();vp.extend(out.tolist());vl.extend(labs.numpy().tolist())
                ema.restore(model)
                vp_cls=reg2cls(vp);vl_int=np.array(vl,dtype=int);vk=qwk(vl_int,vp_cls)

                flag=""
                if vk>best_qwk+ES_D:
                    best_qwk=vk;es=0;flag=" ✅"
                    ema.apply(model);best_state=deepcopy(model.state_dict());ema.restore(model)
                else:es+=1
                print(f"    {pn} Ep{ep+1:02d}/{nep} TrL={tl:.4f} QWK={vk:.4f}{flag} ES={es}/{ES_PAT}")
                cd={"epoch":ep+1,"batch":0,"model_state":model.state_dict(),"best_state":best_state,
                    "optimizer_state":opt.state_dict(),"scheduler_state":sched.state_dict(),
                    "best_qwk":best_qwk,"ema":ema.shadow}
                if scaler:cd["scaler"]=scaler.state_dict()
                torch.save(cd,pc)
                if es>=ES_PAT:print(f"    ⏹ Early stop");break
            del tr_ds,va_ds,tr_ld,va_ld,opt,sched;gc.collect()
            if DEV=="cuda":torch.cuda.empty_cache()

        if best_state:model.load_state_dict(best_state)
        model.eval();oof_f=np.zeros(len(df_va),dtype=np.float32)
        tta=build_tta_tf(384)
        with torch.no_grad():
            for ttf in tta:
                ds2=DRDataset(df_va,ttf,384);ld2=make_loader(ds2,BS[384],shuffle=False)
                bp=[model(im.to(DEV)).cpu().numpy()for im,_ in ld2]
                oof_f+=np.concatenate(bp)
        oof_f/=len(tta);oof_preds[vi]=oof_f;np.save(str(fo),oof_f)
        fold_qwks.append(best_qwk)
        torch.save({"model_state":best_state,"val_qwk":best_qwk,"backbone":CUR_BB,
            "img_size":384,"fold":fold,"ema":ema.shadow},fc)
        mark_done(ff);print(f"  ✅ Fold {fold} QWK={best_qwk:.4f}")
        del model,ema;gc.collect()
        if DEV=="cuda":torch.cuda.empty_cache()

    np.save(str(ART/"oof_preds.npy"),oof_preds);np.save(str(ART/"oof_labels.npy"),oof_labels)
    save_json(fold_qwks,ART/"fold_qwks.json")
    oof_cls=reg2cls(oof_preds);oq=qwk(oof_labels,oof_cls);oa=acc_fn(oof_labels,oof_cls)
    st_save("oof_qwk",float(oq))
    mark_done("training")
    print("\n"+"="*68)
    for i,q in enumerate(fold_qwks):print(f"  Fold {i}: QWK={q:.4f}")
    print(f"  Mean: {np.mean(fold_qwks):.4f} ± {np.std(fold_qwks):.4f}")
    print(f"  OOF QWK={oq:.4f} Acc={oa*100:.2f}%")
    step_end(19,t0)

## Step 19b — Stage 2: Pseudo-Labeling + IDRiD + Messidor


In [ ]:
# ══════════════════════════════════════════════════════════════
# STAGE 2 — Pseudo-labeling (soft) + IDRiD + Messidor
# Rules:
#   - Pseudo labels generated AFTER Stage 1 (Step 19)
#   - IDRiD: label = (gt + pred) / 2
#   - Messidor: label = clip(pred, mean ± 0.5)
#   - NEVER use pseudo labels in validation
# ══════════════════════════════════════════════════════════════

IDRID_DIR   = BASE / "idrid"
MESSIDOR_DIR= BASE / "messidor"
for d in [IDRID_DIR, MESSIDOR_DIR]: d.mkdir(parents=True, exist_ok=True)

def download_idrid():
    """IDRiD: Indian Diabetic Retinopathy Image Dataset — Kaggle/direct."""
    zip_p = IDRID_DIR / "idrid.zip"
    if is_done("idrid_download"): return True
    r = subprocess.run(
        [sys.executable, "-m", "kaggle", "datasets", "download",
         "-d", "mariaherrerot/idrid-dataset", "-p", str(IDRID_DIR)],
        capture_output=True, text=True)
    if r.returncode == 0:
        mark_done("idrid_download"); return True
    # Alternative: try aiphes dataset
    r2 = subprocess.run(
        [sys.executable, "-m", "kaggle", "datasets", "download",
         "-d", "spiyers/idrid", "-p", str(IDRID_DIR)],
        capture_output=True, text=True)
    if r2.returncode == 0:
        mark_done("idrid_download"); return True
    print(f"  ⚠️ IDRiD download failed — skipping IDRiD")
    return False

def download_messidor():
    """Messidor-2 from Kaggle."""
    if is_done("messidor_download"): return True
    r = subprocess.run(
        [sys.executable, "-m", "kaggle", "datasets", "download",
         "-d", "google-brain/messidor2", "-p", str(MESSIDOR_DIR)],
        capture_output=True, text=True)
    if r.returncode == 0:
        mark_done("messidor_download"); return True
    r2 = subprocess.run(
        [sys.executable, "-m", "kaggle", "datasets", "download",
         "-d", "subhajournal/messidor2", "-p", str(MESSIDOR_DIR)],
        capture_output=True, text=True)
    if r2.returncode == 0:
        mark_done("messidor_download"); return True
    print("  ⚠️ Messidor download failed — skipping Messidor")
    return False

def extract_dataset(zip_dir, extract_dir):
    for zp in zip_dir.glob("*.zip"):
        try:
            with zipfile.ZipFile(zp,"r") as zf:
                zf.extractall(extract_dir)
        except Exception as e:
            print(f"  ⚠️ {zp.name}: {e}")

if is_done("stage2_training"):
    step_skip("19b", "STAGE 2 TRAINING")
else:
    t0 = step_start("19b", "STAGE 2: Pseudo-label + IDRiD + Messidor")

    # ── 1. Generate pseudo-labels from Stage 1 ensemble ──
    if not is_done("pseudo_labels"):
        print("  Generating pseudo-labels from Stage 1 ensemble...")
        test_folds = df[df["fold"]==0].reset_index(drop=True)
        ens_pseudo = np.zeros(len(test_folds), dtype=np.float32)
        nm_p = 0
        for fold in range(1, NF):
            fc = CKPT / f"fold{fold}_best.pt"
            if not fc.exists(): continue
            ckpt = safe_load(fc, DEV)
            m_p = build_model(CUR_BB, False)
            m_p.load_state_dict(ckpt["model_state"])
            if "ema" in ckpt:
                for n, p in m_p.named_parameters():
                    if n in ckpt["ema"]: p.data.copy_(ckpt["ema"][n])
            m_p.eval()
            fp = np.zeros(len(test_folds), dtype=np.float32)
            tta_list = build_tta_tf(384)
            with torch.no_grad():
                for ttf in tta_list:
                    ds_p = DRDataset(test_folds, ttf, 384)
                    ld_p = make_loader(ds_p, BS[384], shuffle=False)
                    bp = []
                    for im, _ in ld_p:
                        bp.extend(m_p(im.to(DEV)).cpu().numpy().tolist())
                    fp += np.array(bp)
            fp /= len(tta_list)
            ens_pseudo += np.clip(fp, 0, 4)
            nm_p += 1
            del m_p; gc.collect()
        if nm_p > 0:
            ens_pseudo /= nm_p
            ens_pseudo = np.clip(ens_pseudo, 0, 4)
            # Save soft pseudo-labels
            pseudo_df = test_folds[["id_code","image_path"]].copy()
            pseudo_df["diagnosis"] = ens_pseudo  # soft float
            pseudo_df["source"] = "pseudo"
            pseudo_df.to_parquet(ART/"pseudo_labels.parquet", index=False)
            print(f"  ✅ Pseudo-labels: {len(pseudo_df):,} | mean={ens_pseudo.mean():.3f}")
            mark_done("pseudo_labels")
        else:
            print("  ⚠️ No Stage 1 checkpoints found for pseudo-labeling")

    # ── 2. Download & process IDRiD ──
    df_idrid = None
    if not is_done("idrid_ready"):
        ok_id = download_idrid()
        if ok_id:
            extract_dataset(IDRID_DIR, IDRID_DIR)
            img_map_id = {}
            for p in IDRID_DIR.rglob("*"):
                if p.suffix.lower() in {".jpg",".jpeg",".png"}:
                    img_map_id[p.stem] = str(p)
            # Find grade CSV
            id_csvs = list(IDRID_DIR.rglob("*.csv"))
            df_id = None
            for csv_p in id_csvs:
                try:
                    tmp = pd.read_csv(csv_p)
                    # Look for columns containing grade/level/retinopathy
                    cols = [c.lower() for c in tmp.columns]
                    grade_col = next((tmp.columns[i] for i,c in enumerate(cols)
                                      if any(k in c for k in ["grade","level","retinopathy","dr"])), None)
                    img_col   = next((tmp.columns[i] for i,c in enumerate(cols)
                                      if any(k in c for k in ["image","file","id","name"])), None)
                    if grade_col and img_col:
                        df_id = tmp[[img_col, grade_col]].copy()
                        df_id.columns = ["id_code","diagnosis"]
                        df_id["diagnosis"] = pd.to_numeric(df_id["diagnosis"], errors="coerce")
                        df_id = df_id.dropna(subset=["diagnosis"])
                        df_id["diagnosis"] = df_id["diagnosis"].astype(int).clip(0,4)
                        break
                except: pass
            if df_id is not None:
                df_id["image_path"] = df_id["id_code"].astype(str).map(img_map_id)
                df_id = df_id[df_id["image_path"].notna()].reset_index(drop=True)
                df_id["source"] = "idrid"
                print(f"  IDRiD: {len(df_id):,} images")
                df_idrid = df_id
                mark_done("idrid_ready")
    elif (ART/"idrid_s2.parquet").exists():
        df_idrid = pd.read_parquet(ART/"idrid_s2.parquet")

    # ── 3. Download & process Messidor ──
    df_messidor = None
    if not is_done("messidor_ready"):
        ok_ms = download_messidor()
        if ok_ms:
            extract_dataset(MESSIDOR_DIR, MESSIDOR_DIR)
            img_map_ms = {}
            for p in MESSIDOR_DIR.rglob("*"):
                if p.suffix.lower() in {".jpg",".jpeg",".png",".tif",".tiff"}:
                    img_map_ms[p.stem] = str(p)
            ms_csvs = list(MESSIDOR_DIR.rglob("*.csv"))
            df_ms = None
            for csv_p in ms_csvs:
                try:
                    tmp = pd.read_csv(csv_p)
                    cols = [c.lower() for c in tmp.columns]
                    grade_col = next((tmp.columns[i] for i,c in enumerate(cols)
                                      if any(k in c for k in ["adjudicated","grade","level","retinopathy"])), None)
                    img_col   = next((tmp.columns[i] for i,c in enumerate(cols)
                                      if any(k in c for k in ["image","file","id","name"])), None)
                    if grade_col and img_col:
                        df_ms = tmp[[img_col, grade_col]].copy()
                        df_ms.columns = ["id_code","diagnosis"]
                        df_ms["diagnosis"] = pd.to_numeric(df_ms["diagnosis"], errors="coerce")
                        df_ms = df_ms.dropna(subset=["diagnosis"])
                        df_ms["diagnosis"] = df_ms["diagnosis"].astype(int).clip(0,4)
                        break
                except: pass
            if df_ms is not None:
                df_ms["image_path"] = df_ms["id_code"].astype(str).map(img_map_ms)
                df_ms = df_ms[df_ms["image_path"].notna()].reset_index(drop=True)
                df_ms["source"] = "messidor"
                print(f"  Messidor: {len(df_ms):,} images")
                df_messidor = df_ms
                mark_done("messidor_ready")
    elif (ART/"messidor_s2.parquet").exists():
        df_messidor = pd.read_parquet(ART/"messidor_s2.parquet")

    # ── 4. Apply IDRiD blending: label = (gt + pred) / 2 ──
    def blend_idrid_labels(df_id_in, model_list, tta_list_fn, sz=384):
        preds = np.zeros(len(df_id_in), dtype=np.float32)
        n_m = 0
        for ckpt_p in model_list:
            if not ckpt_p.exists(): continue
            ck = safe_load(ckpt_p, DEV)
            mm = build_model(CUR_BB, False); mm.load_state_dict(ck["model_state"])
            if "ema" in ck:
                for n, p in mm.named_parameters():
                    if n in ck["ema"]: p.data.copy_(ck["ema"][n])
            mm.eval()
            fp2 = np.zeros(len(df_id_in), dtype=np.float32)
            for ttf in tta_list_fn(sz):
                ds2 = DRDataset(df_id_in, ttf, sz); ld2 = make_loader(ds2, BS[sz], shuffle=False)
                bp2 = []
                with torch.no_grad():
                    for im2, _ in ld2:
                        bp2.extend(mm(im2.to(DEV)).cpu().numpy().tolist())
                fp2 += np.array(bp2)
            fp2 /= len(tta_list_fn(sz)); preds += np.clip(fp2, 0, 4); n_m += 1
            del mm; gc.collect()
        if n_m == 0: return df_id_in["diagnosis"].values.astype(float)
        preds /= n_m
        gt = df_id_in["diagnosis"].values.astype(float)
        return np.clip((gt + preds) / 2.0, 0, 4)

    # ── 5. Apply Messidor blending: label = clip(pred, mean ± 0.5) ──
    def blend_messidor_labels(df_ms_in, model_list, tta_list_fn, sz=384):
        preds = np.zeros(len(df_ms_in), dtype=np.float32)
        n_m = 0
        for ckpt_p in model_list:
            if not ckpt_p.exists(): continue
            ck = safe_load(ckpt_p, DEV)
            mm = build_model(CUR_BB, False); mm.load_state_dict(ck["model_state"])
            if "ema" in ck:
                for n, p in mm.named_parameters():
                    if n in ck["ema"]: p.data.copy_(ck["ema"][n])
            mm.eval()
            fp3 = np.zeros(len(df_ms_in), dtype=np.float32)
            for ttf in tta_list_fn(sz):
                ds3 = DRDataset(df_ms_in, ttf, sz); ld3 = make_loader(ds3, BS[sz], shuffle=False)
                bp3 = []
                with torch.no_grad():
                    for im3, _ in ld3:
                        bp3.extend(mm(im3.to(DEV)).cpu().numpy().tolist())
                fp3 += np.array(bp3)
            fp3 /= len(tta_list_fn(sz)); preds += np.clip(fp3, 0, 4); n_m += 1
            del mm; gc.collect()
        if n_m == 0: return df_ms_in["diagnosis"].values.astype(float)
        preds /= n_m
        mean_p = preds.mean()
        return np.clip(preds, mean_p - 0.5, mean_p + 0.5)

    # ── 6. Build Stage 2 training dataframe ──
    ckpt_list = [CKPT / f"fold{f}_best.pt" for f in range(1, NF)]
    df_s2_parts = []

    # Stage 1 train folds (all non-test rows)
    df_s1_train = df[df["fold"] != 0].copy()
    df_s1_train["s2_label"] = df_s1_train["diagnosis"].astype(float)
    df_s2_parts.append(df_s1_train[["id_code","image_path","s2_label","source"]].copy()
                       if "source" in df_s1_train.columns
                       else df_s1_train[["id_code","image_path","s2_label"]].assign(source="aptos2019"))

    # Add pseudo-labels (test set, soft)
    if (ART/"pseudo_labels.parquet").exists():
        df_ps = pd.read_parquet(ART/"pseudo_labels.parquet")
        df_ps["s2_label"] = df_ps["diagnosis"].astype(float)  # soft
        df_s2_parts.append(df_ps[["id_code","image_path","s2_label"]].assign(source="pseudo"))
        print(f"  + Pseudo-labels: {len(df_ps):,}")

    # Add IDRiD with blended labels
    if df_idrid is not None and len(df_idrid) > 0:
        print("  Computing IDRiD blended labels...")
        df_idrid["s2_label"] = blend_idrid_labels(df_idrid, ckpt_list, build_tta_tf, 384)
        df_idrid.to_parquet(ART/"idrid_s2.parquet", index=False)
        df_s2_parts.append(df_idrid[["id_code","image_path","s2_label","source"]])
        print(f"  + IDRiD: {len(df_idrid):,} (blended labels)")

    # Add Messidor with clipped labels
    if df_messidor is not None and len(df_messidor) > 0:
        print("  Computing Messidor blended labels...")
        df_messidor["s2_label"] = blend_messidor_labels(df_messidor, ckpt_list, build_tta_tf, 384)
        df_messidor.to_parquet(ART/"messidor_s2.parquet", index=False)
        df_s2_parts.append(df_messidor[["id_code","image_path","s2_label","source"]])
        print(f"  + Messidor: {len(df_messidor):,} (blended labels)")

    df_stage2 = pd.concat(df_s2_parts, ignore_index=True)
    df_stage2["diagnosis"] = df_stage2["s2_label"].clip(0,4)  # soft float
    df_stage2.to_parquet(ART/"df_stage2.parquet", index=False)
    print(f"  Stage 2 total: {len(df_stage2):,} images")

    # ── 7. Stage 2 Training Loop ──
    # Val set = APTOS fold 0 ONLY (NEVER pseudo-labels)
    df_val_s2 = df[df["fold"]==0].reset_index(drop=True)

    PHASES_S2 = [
        {"sz":384, "ep":30, "name":"S2-P1", "unf":4,  "lr":2e-5, "acc":2},
        {"sz":512, "ep":20, "name":"S2-P2", "unf":99, "lr":5e-6, "acc":4},
    ]
    ES_S2 = 8
    best_s2_qwk = float(st_get("oof_qwk", 0.0))
    crit = nn.SmoothL1Loss()

    for fold in range(1, NF):
        ff_s2 = f"s2_fold{fold}"
        fc_s2 = CKPT / f"s2_fold{fold}_best.pt"
        if is_done(ff_s2) and fc_s2.exists():
            ck = safe_load(fc_s2, "cpu")
            print(f"  ✅ [S2 RESUME] Fold {fold} QWK={ck.get('val_qwk',0):.4f}"); continue

        seed_everything(SEED + fold + 100)
        # Load Stage 1 best as init
        fc_s1 = CKPT / f"fold{fold}_best.pt"
        model = build_model(CUR_BB, False)
        if fc_s1.exists():
            ck_s1 = safe_load(fc_s1, DEV)
            model.load_state_dict(ck_s1["model_state"])
            if "ema" in ck_s1:
                for n, p in model.named_parameters():
                    if n in ck_s1["ema"]: p.data.copy_(ck_s1["ema"][n])
            print(f"  ↻ S2 Fold {fold}: init from S1 (QWK={ck_s1.get('val_qwk',0):.4f})")
        model.to(DEV); ema_s2 = EMA(model); best_s2_q = -1.0; best_s2_state = None

        # Stage 2 uses ALL stage2 data for training fold
        # (no fold split needed — pseudo/IDRiD/Messidor are extra, val stays APTOS fold0)
        df_tr_s2 = df_stage2.copy()

        for pi, ph in enumerate(PHASES_S2):
            sz=ph["sz"]; nep=ph["ep"]; pn=ph["name"]; unf=ph["unf"]
            plr=ph["lr"]; accum=ph["acc"]; bs=BS[sz]
            pc_s2 = CKPT / f"s2_fold{fold}_p{pi}.pt"

            if unf == 0:   model.freeze_backbone()
            elif unf >= 99: model.unfreeze_all()
            else:           model.unfreeze_top(unf)

            tr_ds_s2 = DRDataset(df_tr_s2, build_train_tf(sz), sz, use_cache=False)
            va_ds_s2 = DRDataset(df_val_s2, build_val_tf(sz), sz)
            smp_s2   = build_sampler(df_tr_s2.assign(diagnosis=df_tr_s2["diagnosis"].round().astype(int)))
            tr_ld_s2 = make_loader(tr_ds_s2, bs, sampler=smp_s2, drop_last=True)
            va_ld_s2 = make_loader(va_ds_s2, bs, shuffle=False)

            opt_s2 = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()), lr=plr, weight_decay=WD)
            _w2=3; _n2=nep
            def _lr2(ep, warmup=_w2, total=_n2):
                if ep < warmup: return (ep+1)/warmup
                return 0.5*(1+np.cos(np.pi*(ep-warmup)/(total-warmup)))
            sched_s2 = torch.optim.lr_scheduler.LambdaLR(opt_s2, _lr2)
            scaler_s2 = torch.amp.GradScaler("cuda") if AMP else None

            ep_start=0; batch_start=0; es=0
            if pc_s2.exists():
                ps = safe_load(pc_s2, DEV)
                model.load_state_dict(ps["model_state"])
                opt_s2.load_state_dict(ps["optimizer_state"])
                sched_s2.load_state_dict(ps["scheduler_state"])
                ep_start = ps["epoch"]; best_s2_q = ps.get("best_qwk", best_s2_q)
                if ps.get("best_state"): best_s2_state = ps["best_state"]
                if ps.get("ema"): ema_s2.shadow = ps["ema"]
                batch_start = ps.get("batch", 0)
                if scaler_s2 and ps.get("scaler"): scaler_s2.load_state_dict(ps["scaler"])

            for ep in range(ep_start, nep):
                model.train(); el=0.0; nb=0; opt_s2.zero_grad()
                for step, (imgs, labs) in enumerate(tr_ld_s2):
                    if ep == ep_start and step < batch_start: continue
                    imgs, labs = imgs.to(DEV), labs.to(DEV)
                    if AMP:
                        with torch.amp.autocast("cuda"):
                            out = model(imgs); loss = crit(out, labs) / accum
                        scaler_s2.scale(loss).backward()
                        if (step+1) % accum == 0:
                            scaler_s2.unscale_(opt_s2)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                            scaler_s2.step(opt_s2); scaler_s2.update(); opt_s2.zero_grad()
                    else:
                        out = model(imgs); loss = crit(out, labs) / accum; loss.backward()
                        if (step+1) % accum == 0:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                            opt_s2.step(); opt_s2.zero_grad()
                    el += loss.item() * accum; nb += 1; ema_s2.update(model)
                batch_start=0; sched_s2.step(); tl=el/max(nb,1)

                ema_s2.apply(model); model.eval(); vp2=[]; vl2=[]
                with torch.no_grad():
                    for imgs2, labs2 in va_ld_s2:
                        out2 = model(imgs2.to(DEV)).cpu().numpy()
                        vp2.extend(out2.tolist()); vl2.extend(labs2.numpy().tolist())
                ema_s2.restore(model)
                vp2_cls = reg2cls(vp2); vl2_int = np.array(vl2, dtype=int)
                vk2 = qwk(vl2_int, vp2_cls)
                flag = ""
                if vk2 > best_s2_q + ES_D:
                    best_s2_q = vk2; es = 0; flag = " ✅"
                    ema_s2.apply(model)
                    best_s2_state = deepcopy(model.state_dict())
                    ema_s2.restore(model)
                else: es += 1
                print(f"    S2 Fold{fold} {pn} Ep{ep+1:02d}/{nep} L={tl:.4f} QWK={vk2:.4f}{flag} ES={es}/{ES_S2}")
                cd = {"epoch":ep+1,"batch":0,"model_state":model.state_dict(),
                      "best_state":best_s2_state,"optimizer_state":opt_s2.state_dict(),
                      "scheduler_state":sched_s2.state_dict(),"best_qwk":best_s2_q,"ema":ema_s2.shadow}
                if scaler_s2: cd["scaler"] = scaler_s2.state_dict()
                torch.save(cd, pc_s2)
                if es >= ES_S2: print(f"    ⏹ S2 Early stop"); break
            del tr_ds_s2, va_ds_s2, tr_ld_s2, va_ld_s2, opt_s2, sched_s2; gc.collect()
            if DEV=="cuda": torch.cuda.empty_cache()

        if best_s2_state: model.load_state_dict(best_s2_state)
        torch.save({"model_state": best_s2_state, "val_qwk": best_s2_q,
                    "backbone": CUR_BB, "fold": fold, "ema": ema_s2.shadow}, fc_s2)
        mark_done(ff_s2)
        print(f"  ✅ S2 Fold {fold} QWK={best_s2_q:.4f}")
        del model, ema_s2; gc.collect()
        if DEV=="cuda": torch.cuda.empty_cache()

    st_save("stage2_best_qwk", best_s2_qwk)
    mark_done("stage2_training"); step_end("19b", t0)


## Step 20 — Validation


In [ ]:
if is_done("training"):
    t0=step_start(20,"VALIDATION")
    fq=load_json(ART/"fold_qwks.json")
    for i,q in enumerate(fq):print(f"  Fold {i}: QWK={q:.4f}{' ← best'if i==int(np.argmax(fq))else''}")
    print(f"  Mean: {np.mean(fq):.4f} ± {np.std(fq):.4f}")
    print(f"  OOF QWK: {st_get('oof_qwk','N/A')}")
    mark_done("validation");step_end(20,t0)
else:print("  ⚠️ Run Step 19 first")

## Step 21 — Early Stopping


In [ ]:
t0=step_start(21,"EARLY STOPPING")
print(f"  patience = {ES_PAT}")
print(f"  min_delta = {ES_D}")
print(f"  Monitor: Validation QWK (via EMA weights)")
print(f"  Integrated in Step 19 training loop")
mark_done("early_stopping");step_end(21,t0)

## Step 22 — OOF Predictions


In [ ]:
if is_done("training"):
    t0=step_start(22,"OOF PREDICTIONS")
    op=np.load(str(ART/"oof_preds.npy"));ol=np.load(str(ART/"oof_labels.npy"))
    oc=reg2cls(op)
    print(f"  Shape: {op.shape}")
    print(f"  OOF QWK:  {qwk(ol,oc):.4f}")
    print(f"  OOF Acc:  {acc_fn(ol,oc)*100:.2f}%")
    print(f"  Mean pred per class:")
    for g in range(5):
        mask=ol==g
        if mask.sum()>0:print(f"    G{g}: mean={op[mask].mean():.3f}")
    mark_done("oof");step_end(22,t0)
else:print("  ⚠️ Run Step 19 first")

## Step 23 — TTA (Test-Time Augmentation)


In [ ]:
t0=step_start(23,"TTA")
print("  5-view TTA:")
print("    1. Original (resize + normalize)")
print("    2. Horizontal flip")
print("    3. Rotate +10°")
print("    4. Rotate -10°")
print("    5. Vertical flip")
print("  Strategy: Average raw regression predictions across views")
print("  Applied in: OOF (Step 19) + Testing (Step 25)")
mark_done("tta");step_end(23,t0)

## Step 24 — Threshold Optimization


In [ ]:
if is_done("thresholds"):
    od=load_json(ART/"thresholds.json");step_skip(24,"THRESHOLDS",f"QWK={od.get('qwk')}")
else:
    t0=step_start(24,"THRESHOLD OPTIMIZATION")
    op=np.load(str(ART/"oof_preds.npy"));ol=np.load(str(ART/"oof_labels.npy"))
    # 1. Default
    base_cls=reg2cls(op);base_q=qwk(ol,base_cls)
    print(f"  Default {THRESH}: QWK={base_q:.4f}")
    # 2. Optimize via Nelder-Mead
    def neg_qwk_t(t):
        c=np.digitize(np.clip(op,0,4),sorted(t));return-qwk(ol,c)
    res=minimize(neg_qwk_t,THRESH,method="Nelder-Mead",options={"maxiter":2000})
    opt_t=sorted(res.x.tolist());opt_cls=np.digitize(np.clip(op,0,4),opt_t);opt_q=qwk(ol,opt_cls)
    print(f"  Optimized {[round(t,3)for t in opt_t]}: QWK={opt_q:.4f}")
    # 3. FORCE FINAL per winner
    final_t=opt_t if opt_q>base_q else THRESH
    final_q=max(opt_q,base_q)
    print(f"  ✅ Final: {[round(t,3)for t in final_t]}: QWK={final_q:.4f}")
    save_json({"default":THRESH,"optimized":opt_t,"final":final_t,"qwk":float(final_q)},ART/"thresholds.json")
    st_save("opt_qwk",float(final_q));mark_done("thresholds");step_end(24,t0)

## Step 25 — Final Testing (Ensemble + TTA)


In [ ]:
if is_done("test"):step_skip(25,"TESTING",f"QWK={st_get('test_qwk')}")
else:
    t0=step_start(25,"FINAL TESTING")
    df_test=df[df["fold"]==0].reset_index(drop=True)
    print(f"  Test set: {len(df_test):,} (fold 0, never trained)")
    ens=np.zeros(len(df_test),dtype=np.float32);nm=0
    for fold in range(1,NF):
        fc=CKPT/f"fold{fold}_best.pt"
        if not fc.exists():print(f"  ⚠️ fold{fold} missing");continue
        ckpt=safe_load(fc,DEV);model=build_model(CUR_BB,False)
        model.load_state_dict(ckpt["model_state"])
        if "ema" in ckpt:
            for n,p in model.named_parameters():
                if n in ckpt["ema"]:p.data.copy_(ckpt["ema"][n])
        model.eval()
        fp=np.zeros(len(df_test),dtype=np.float32)
        tta=build_tta_tf(384)
        with torch.no_grad():
            for ttf in tta:
                ds=DRDataset(df_test,ttf,384);ld=make_loader(ds,BS[384],shuffle=False)
                bp=[]
                for im,_ in ld:bp.extend(model(im.to(DEV)).cpu().numpy().tolist())
                fp+=np.array(bp)
        fp/=len(tta);fp=np.clip(fp,0,4)
        ens+=fp;nm+=1
        fq_v=qwk(df_test["diagnosis"].values,reg2cls(fp))
        print(f"  Fold {fold}: QWK={fq_v:.4f}")
        del model;gc.collect()
    if nm>0:
        ens/=nm;ens=np.clip(ens,0,4)
        tl=df_test["diagnosis"].values;tp=reg2cls(ens)
        tq=qwk(tl,tp);ta=acc_fn(tl,tp)
        print(f"\n  ✅ Ensemble ({nm} models × 5 TTA)")
        print(f"     Test QWK: {tq:.4f}")
        print(f"     Test Acc: {ta*100:.2f}%")
        np.save(str(ART/"test_preds.npy"),ens)
        st_save("test_qwk",float(tq));st_save("test_acc",float(ta))
    mark_done("test");step_end(25,t0)

## Step 25b — Stage 2 Final Testing (Ensemble + TTA)


In [ ]:
# Use Stage 2 checkpoints (if available) for final ensemble
if any((CKPT/f"s2_fold{f}_best.pt").exists() for f in range(1,NF)):
    t0 = step_start("25b", "STAGE 2 FINAL TESTING")
    df_test = df[df["fold"]==0].reset_index(drop=True)
    print(f"  Test set: {len(df_test):,} (fold 0, never trained on)")
    ens_s2 = np.zeros(len(df_test), dtype=np.float32); nm_s2 = 0
    for fold in range(1, NF):
        fc2 = CKPT / f"s2_fold{fold}_best.pt"
        if not fc2.exists(): print(f"  ⚠️ s2_fold{fold} missing"); continue
        ckpt2 = safe_load(fc2, DEV)
        m2 = build_model(CUR_BB, False); m2.load_state_dict(ckpt2["model_state"])
        if "ema" in ckpt2:
            for n, p in m2.named_parameters():
                if n in ckpt2["ema"]: p.data.copy_(ckpt2["ema"][n])
        m2.eval()
        fp2 = np.zeros(len(df_test), dtype=np.float32)
        tta_list = build_tta_tf(384)
        with torch.no_grad():
            for ttf in tta_list:
                ds2 = DRDataset(df_test, ttf, 384); ld2 = make_loader(ds2, BS[384], shuffle=False)
                bp2 = []
                for im2, _ in ld2: bp2.extend(m2(im2.to(DEV)).cpu().numpy().tolist())
                fp2 += np.array(bp2)
        fp2 /= len(tta_list); fp2 = np.clip(fp2, 0, 4); ens_s2 += fp2; nm_s2 += 1
        fq2 = qwk(df_test["diagnosis"].values, reg2cls(fp2))
        print(f"  S2 Fold {fold}: QWK={fq2:.4f}")
        del m2; gc.collect()
    if nm_s2 > 0:
        ens_s2 /= nm_s2; ens_s2 = np.clip(ens_s2, 0, 4)
        tl2 = df_test["diagnosis"].values; tp2 = reg2cls(ens_s2)
        tq2 = qwk(tl2, tp2); ta2 = acc_fn(tl2, tp2)
        print(f"\n  ✅ Stage 2 Ensemble ({nm_s2} models × 5 TTA)")
        print(f"     Test QWK: {tq2:.4f}")
        print(f"     Test Acc: {ta2*100:.2f}%")
        np.save(str(ART/"s2_test_preds.npy"), ens_s2)
        st_save("s2_test_qwk", float(tq2)); st_save("s2_test_acc", float(ta2))
    step_end("25b", t0)
else:
    print("  Stage 2 checkpoints not found — skipping Step 25b")


## Step 26 — Metrics (QWK, Accuracy, Precision, Recall, F1)


In [ ]:
if is_done("metrics"):step_skip(26,"METRICS")
else:
    t0=step_start(26,"METRICS")
    try:get_ipython().run_line_magic('matplotlib','inline')
    except:pass
    op=np.load(str(ART/"oof_preds.npy"));ol=np.load(str(ART/"oof_labels.npy"))
    pred=reg2cls(op);oq=qwk(ol,pred);oa=acc_fn(ol,pred)
    pr=precision_score(ol,pred,average="weighted",zero_division=0)
    rc=recall_score(ol,pred,average="weighted",zero_division=0)
    f1v=f1_score(ol,pred,average="weighted",zero_division=0)
    m={"qwk":round(oq,4),"accuracy":round(oa,4),"precision":round(pr,4),"recall":round(rc,4),"f1":round(f1v,4)}
    for k,v in m.items():print(f"  {k:12s}: {v}")
    # Confusion Matrix
    fig,ax=plt.subplots(1,2,figsize=(16,6))
    cm=confusion_matrix(ol,pred)
    ConfusionMatrixDisplay(cm,display_labels=[f"G{i}"for i in range(5)]).plot(ax=ax[0],colorbar=False,cmap="Blues")
    ax[0].set_title(f"QWK={oq:.4f} Acc={oa*100:.1f}%",fontweight="bold")
    pcr=cm.diagonal()/np.maximum(cm.sum(axis=1),1)
    ax[1].bar([GRADE[i]for i in range(5)],pcr,color=GCOL,edgecolor="k")
    ax[1].axhline(0.85,color="red",ls="--",label="85%");ax[1].set_ylim(0,1.05);ax[1].legend()
    ax[1].set_title("Per-Class Recall",fontweight="bold")
    for i,v in enumerate(pcr):ax[1].text(i,v+0.01,f"{v:.2f}",ha="center")
    plt.tight_layout();plt.savefig(PLOT/"metrics.png",dpi=150);plt.show()
    print(f"\n{classification_report(ol,pred,target_names=[GRADE[i]for i in range(5)])}")
    save_json(m,LOG/"metrics.json");pd.DataFrame([m]).to_csv(LOG/"metrics.csv",index=False)
    mark_done("metrics");step_end(26,t0)

## Step 27 — Grad-CAM++ Explainability


In [ ]:
if is_done("explainability"):step_skip(27,"GRAD-CAM++")
else:
    t0=step_start(27,"GRAD-CAM++")
    try:get_ipython().run_line_magic('matplotlib','inline')
    except:pass
    try:
        from pytorch_grad_cam import GradCAMPlusPlus
        from pytorch_grad_cam.utils.image import show_cam_on_image
        from pytorch_grad_cam.utils.model_targets import RawScoresOutputTarget
        fq=load_json(ART/"fold_qwks.json");bf=int(np.argmax(fq))
        ckpt=safe_load(CKPT/f"fold{bf}_best.pt","cpu")
        gm=build_model(CUR_BB,False);gm.load_state_dict(ckpt["model_state"]);gm.eval().to("cpu")
        # Find last conv
        tl=None
        for m in reversed(list(gm.backbone.modules())):
            if isinstance(m,nn.Conv2d):tl=[m];break
        if tl is None:tl=[list(gm.backbone.children())[-1]]
        cam=GradCAMPlusPlus(model=gm,target_layers=tl)
        fig,axes=plt.subplots(5,4,figsize=(14,15))
        for g in range(5):
            samp=df[df["diagnosis"]==g].sample(min(2,(df["diagnosis"]==g).sum()),random_state=SEED)
            for j,(_,row)in enumerate(samp.iterrows()):
                img=preprocess_fundus(row["image_path"],384)
                inp=build_val_tf(384)(image=img)["image"].unsqueeze(0)
                gs=cam(input_tensor=inp,targets=[RawScoresOutputTarget()])
                ci=show_cam_on_image(img.astype(np.float32)/255.,gs[0],use_rgb=True)
                axes[g][j*2].imshow(img);axes[g][j*2].axis("off")
                if j==0:axes[g][0].set_ylabel(f"G{g}",fontsize=11,fontweight="bold",rotation=0,labelpad=30,va="center")
                axes[g][j*2+1].imshow(ci);axes[g][j*2+1].axis("off")
        plt.suptitle(f"Grad-CAM++ — Fold {bf} (QWK={fq[bf]:.4f})",fontweight="bold")
        plt.tight_layout();plt.savefig(PLOT/"gradcam.png",dpi=130);plt.show()
        del gm;gc.collect()
    except ImportError:print("  pip install grad-cam")
    except Exception as e:print(f"  ⚠️ {e}")
    mark_done("explainability");step_end(27,t0)

## Step 28 — Model Export


In [ ]:
import shutil as _sh
if is_done("export"):step_skip(28,"EXPORT")
else:
    t0=step_start(28,"MODEL EXPORT")
    EXPORT.mkdir(exist_ok=True)
    fq=load_json(ART/"fold_qwks.json");bf=int(np.argmax(fq))
    src=CKPT/f"fold{bf}_best.pt"
    if src.exists():
        _sh.copy2(src,EXPORT/"best_model.pt");print(f"  best_model.pt (fold {bf})")
        ckpt=safe_load(src,"cpu")
        if "ema" in ckpt:
            em=build_model(CUR_BB,False)
            for n,p in em.named_parameters():
                if n in ckpt["ema"]:p.data.copy_(ckpt["ema"][n])
            torch.save({"model_state":em.state_dict()},EXPORT/"ema_model.pt")
            print("  ema_model.pt");del em
    thr=load_json(ART/"thresholds.json")if(ART/"thresholds.json").exists()else{"final":THRESH}
    save_json(thr,EXPORT/"thresholds.json")
    save_json({str(k):v for k,v in GRADE.items()},EXPORT/"labels.json")
    save_json({"backbone":CUR_BB,"approach":"regression","thresholds":THRESH,
        "fold_qwks":fq,"oof_qwk":st_get("oof_qwk"),"test_qwk":st_get("test_qwk")},EXPORT/"metadata.json")
    mark_done("export");step_end(28,t0)

## Step 29 — Deployment (Streamlit + Validator)


In [ ]:
import shutil as _sh
if is_done("deployment"):step_skip(29,"DEPLOYMENT")
else:
    t0=step_start(29,"DEPLOYMENT")
    DEPLOY.mkdir(exist_ok=True)
    for f in EXPORT.glob("*"):
        if f.is_file():_sh.copy2(f,DEPLOY/f.name)

    mu=[]
    mu.append("import numpy as np,cv2,torch,torch.nn as nn,torch.nn.functional as F,timm")
    mu.append("import albumentations as A;from albumentations.pytorch import ToTensorV2")
    mu.append("IMEAN=[0.485,0.456,0.406];ISTD=[0.229,0.224,0.225]")
    mu.append("GRADE={0:'No DR',1:'Mild',2:'Moderate',3:'Severe',4:'Proliferative'}")
    mu.append("THR=[0.7,1.5,2.5,3.5];SZ=384;BB='tf_efficientnetv2_b1'")
    mu.append("class GeM(nn.Module):")
    mu.append("    def __init__(self,p=3,eps=1e-6):")
    mu.append("        super().__init__();self.p=nn.Parameter(torch.ones(1)*p);self.eps=eps")
    mu.append("    def forward(self,x):return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),(x.size(-2),x.size(-1))).pow(1./self.p)")
    mu.append("class DRModel(nn.Module):")
    mu.append("    def __init__(self,bb=BB,drop=0.5):")
    mu.append("        super().__init__()")
    mu.append("        self.backbone=timm.create_model(bb,pretrained=False,num_classes=0,global_pool='')")
    mu.append("        fd=self.backbone.num_features;self.pool=GeM()")
    mu.append("        self.head=nn.Sequential(nn.Flatten(),nn.Linear(fd,256),nn.BatchNorm1d(256),nn.ReLU(True),nn.Dropout(drop),nn.Linear(256,1))")
    mu.append("    def forward(self,x):return self.head(self.pool(self.backbone(x))).squeeze(-1)")
    (DEPLOY/"model_utils.py").write_text("\n".join(mu))
    print("  model_utils.py")

    vl=[]
    vl.append("import cv2,numpy as np")
    vl.append("def is_retinal(img):")
    vl.append("    if img.shape[0]<100 or img.shape[1]<100:return False,'Too small'")
    vl.append("    g=cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)")
    vl.append("    if float(g.mean())<5:return False,'Black'")
    vl.append("    if float(g.mean())>250:return False,'White'")
    vl.append("    return True,'Valid'  # NEVER reject real retinal — fail-safe")
    (DEPLOY/"validator.py").write_text("\n".join(vl))
    print("  validator.py (fail-safe)")

    # ── Full Streamlit app.py ──
    app_lines = []
    app_lines.append("import streamlit as st")
    app_lines.append("import numpy as np, cv2, torch, json")
    app_lines.append("from pathlib import Path")
    app_lines.append("from PIL import Image")
    app_lines.append("import albumentations as A")
    app_lines.append("from albumentations.pytorch import ToTensorV2")
    app_lines.append("import sys; sys.path.insert(0, str(Path(__file__).parent))")
    app_lines.append("from model_utils import DRModel, GRADE, THR, SZ, BB, IMEAN, ISTD")
    app_lines.append("from validator import is_retinal")
    app_lines.append("")
    app_lines.append("st.set_page_config(page_title='DR Grading', page_icon='👁️', layout='centered')")
    app_lines.append("st.title('🩺 Diabetic Retinopathy Grading')")
    app_lines.append("st.markdown('**Upload a fundus photograph to get an automated DR grade.**')")
    app_lines.append("st.warning('⚠️ RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT')")
    app_lines.append("")
    app_lines.append("MODEL_PATH = Path(__file__).parent / 'best_model.pt'")
    app_lines.append("THR_PATH   = Path(__file__).parent / 'thresholds.json'")
    app_lines.append("")
    app_lines.append("@st.cache_resource")
    app_lines.append("def load_model():")
    app_lines.append("    device = 'cuda' if torch.cuda.is_available() else 'cpu'")
    app_lines.append("    model = DRModel(bb=BB)")
    app_lines.append("    if MODEL_PATH.exists():")
    app_lines.append("        ck = torch.load(str(MODEL_PATH), map_location=device, weights_only=False)")
    app_lines.append("        sd = ck.get('model_state', ck)")
    app_lines.append("        model.load_state_dict(sd, strict=False)")
    app_lines.append("    model.eval().to(device)")
    app_lines.append("    return model, device")
    app_lines.append("")
    app_lines.append("def load_thresholds():")
    app_lines.append("    if THR_PATH.exists():")
    app_lines.append("        thr_data = json.loads(THR_PATH.read_text())")
    app_lines.append("        return thr_data.get('final', THR)")
    app_lines.append("    return THR")
    app_lines.append("")
    app_lines.append("def preprocess(img_np, sz=SZ):")
    app_lines.append("    tf = A.Compose([A.Resize(sz,sz), A.Normalize(mean=IMEAN,std=ISTD), ToTensorV2()])")
    app_lines.append("    return tf(image=img_np)['image'].unsqueeze(0)")
    app_lines.append("")
    app_lines.append("def tta_predict(model, img_np, device, sz=SZ):")
    app_lines.append("    n = A.Normalize(mean=IMEAN, std=ISTD); r = A.Resize(sz,sz)")
    app_lines.append("    tfs = [")
    app_lines.append("        A.Compose([r, n, ToTensorV2()]),")
    app_lines.append("        A.Compose([A.HorizontalFlip(p=1), r, n, ToTensorV2()]),")
    app_lines.append("        A.Compose([A.Rotate(limit=(10,10),p=1), r, n, ToTensorV2()]),")
    app_lines.append("        A.Compose([A.Rotate(limit=(-10,-10),p=1), r, n, ToTensorV2()]),")
    app_lines.append("        A.Compose([A.VerticalFlip(p=1), r, n, ToTensorV2()]),")
    app_lines.append("    ]")
    app_lines.append("    preds = []")
    app_lines.append("    with torch.no_grad():")
    app_lines.append("        for tf in tfs:")
    app_lines.append("            inp = tf(image=img_np)['image'].unsqueeze(0).to(device)")
    app_lines.append("            preds.append(model(inp).item())")
    app_lines.append("    return float(np.clip(np.mean(preds), 0, 4))")
    app_lines.append("")
    app_lines.append("def apply_thresholds(raw, thr):")
    app_lines.append("    return int(np.digitize(np.clip(raw, 0, 4), sorted(thr)))")
    app_lines.append("")
    app_lines.append("uploaded = st.file_uploader('Upload fundus image', type=['png','jpg','jpeg','tiff'])")
    app_lines.append("")
    app_lines.append("if uploaded:")
    app_lines.append("    file_bytes = np.asarray(bytearray(uploaded.read()), dtype=np.uint8)")
    app_lines.append("    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)")
    app_lines.append("    if img_bgr is None:")
    app_lines.append("        st.error('Could not decode image.'); st.stop()")
    app_lines.append("    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)")
    app_lines.append("    col1, col2 = st.columns(2)")
    app_lines.append("    with col1: st.image(img_rgb, caption='Uploaded Image', use_container_width=True)")
    app_lines.append("    valid, reason = is_retinal(img_rgb)")
    app_lines.append("    if not valid:")
    app_lines.append("        st.warning(f'Validation note: {reason} — proceeding anyway (fail-safe)')")
    app_lines.append("    with st.spinner('Grading...'):")
    app_lines.append("        model, device = load_model()")
    app_lines.append("        thr = load_thresholds()")
    app_lines.append("        img_resized = cv2.resize(img_rgb, (SZ, SZ))")
    app_lines.append("        raw_pred = tta_predict(model, img_resized, device)")
    app_lines.append("        grade = apply_thresholds(raw_pred, thr)")
    app_lines.append("        grade_name = GRADE.get(grade, 'Unknown')")
    app_lines.append("    with col2:")
    app_lines.append("        COLORS = {0:'#2ecc71',1:'#f1c40f',2:'#e67e22',3:'#e74c3c',4:'#8e44ad'}")
    app_lines.append("        color = COLORS.get(grade,'#888')")
    app_lines.append("        st.markdown(f'<div style="background:{color};color:white;padding:20px;border-radius:10px;text-align:center">'")
    app_lines.append("                    f'<h2>Grade {grade}</h2><h3>{grade_name}</h3>'")
    app_lines.append("                    f'<p>Raw score: {raw_pred:.3f}</p></div>', unsafe_allow_html=True)")
    app_lines.append("    st.markdown('---')")
    app_lines.append("    st.markdown('**DR Grading Scale:**')")
    app_lines.append("    for g, name in GRADE.items():")
    app_lines.append("        marker = ' ← **Current**' if g == grade else ''")
    app_lines.append("        st.markdown(f'- **Grade {g}**: {name}{marker}')")
    app_lines.append("else:")
    app_lines.append("    st.info('Please upload a fundus photograph to begin grading.')")
    (DEPLOY/"app.py").write_text("\n".join(app_lines))
    print("  app.py (full Streamlit app with TTA + thresholds + fail-safe validator)")

    (DEPLOY/"requirements.txt").write_text(
        "torch>=2.1\ntorchvision\ntimm>=1.0.0\nalbumentations>=1.4.0\n"
        "opencv-python-headless\nstreamlit\nnumpy\nPillow\n")
    print("  requirements.txt")
    (DEPLOY/"README.md").write_text(
        "# DR Grading v23\n\n## Run\n```\nstreamlit run app.py\n```\n\n"
        "**RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT**\n")
    print("  README.md")

    mark_done("deployment");step_end(29,t0)


## Step 30 — Inference Pipeline + Final Summary


In [ ]:
if not is_done("inference"):mark_done("inference")
t0=step_start(30,"INFERENCE PIPELINE + FINAL SUMMARY")

print("  ── Inference Pipeline (strict order) ──")
print("  1. Preprocess (resize only)")
print("  2. TTA (5 views)")
print("  3. Average predictions")
print("  4. clip [0, 4]")
print("  5. Apply thresholds [0.7, 1.5, 2.5, 3.5]")
print("  6. Output class 0-4")

state={}
if _MS.exists():
    try:state=json.loads(_MS.read_text())
    except:pass

print(f"\n  {'='*50}")
print(f"  🩺 FINAL SUMMARY")
print(f"  {'='*50}")
print(f"  Strategy   : Regression + SmoothL1 (1st-place replication)")
print(f"  Backbone   : {CUR_BB}")
print(f"  Device     : {DEV.upper()}")
print(f"  Dataset    : {len(df):,} images | {NC} classes")
print(f"  Thresholds : {THRESH}")
print(f"  EMA        : decay=0.9999")
print(f"  Models     : {len(BACKBONES)} defined ({len(set(b[0]for b in BACKBONES))} backbones × 2 seeds)")
print()

if(ART/"fold_qwks.json").exists():
    fq=load_json(ART/"fold_qwks.json")
    print("  ── K-Fold Results ──")
    for i,q in enumerate(fq):print(f"    Fold {i}: QWK = {q:.4f}{' ← best'if i==int(np.argmax(fq))else''}")
    print(f"    Mean   : {np.mean(fq):.4f} ± {np.std(fq):.4f}")

print(f"\n  OOF QWK    : {state.get('oof_qwk','N/A')}")
print(f"  Opt QWK    : {state.get('opt_qwk','N/A')}")
print(f"  Test QWK   : {state.get('test_qwk','N/A')}")
ta=state.get('test_acc')
print(f"  Test Acc   : {float(ta)*100:.2f}%"if ta else"  Test Acc   : N/A")

if(LOG/"metrics.json").exists():
    m=load_json(LOG/"metrics.json")
    print(f"\n  ── Detailed Metrics ──")
    print(f"  Precision  : {m.get('precision','N/A')}")
    print(f"  Recall     : {m.get('recall','N/A')}")
    print(f"  F1 Score   : {m.get('f1','N/A')}")

done=sorted(FLAG.glob("*.done"))
print(f"\n  ✅ {len(done)} steps completed")
print(f"  Export     : {EXPORT}")
print(f"  Deploy     : {DEPLOY}")
print(f"  {'='*50}")
print(f"  ⚠️ RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT")
print(f"  {'='*50}")
step_end(30,t0)